In [2]:
#analysing employee dataset and creating api(s) for the data

In [3]:
import pandas as pd

df = pd.read_csv('Messy_Employee_dataset.csv')
null_values = df.isnull().sum()
print(null_values)

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64


In [4]:
df = df.dropna(subset=['Salary'])
df = df.dropna(subset=['Age'])
print(df)

df['Join_Date'] = pd.to_datetime(df['Join_Date'])

df['Join_Year'] = df['Join_Date'].dt.year
df['Join_Month'] = df['Join_Date'].dt.month

     Employee_ID First_Name Last_Name   Age   Department_Region    Status  \
0        EMP1000        Bob     Davis  25.0   DevOps-California    Active   
3        EMP1003        Eva     Davis  25.0        Admin-Nevada  Inactive   
4        EMP1004      Frank  Williams  25.0  Cloud Tech-Florida    Active   
5        EMP1005      Alice    Garcia  40.0         Sales-Texas  Inactive   
7        EMP1007        Bob     Jones  30.0  Cloud Tech-Florida  Inactive   
...          ...        ...       ...   ...                 ...       ...   
1014     EMP2014      Grace     Jones  30.0      Finance-Nevada   Pending   
1016     EMP2016      David   Johnson  30.0    Cloud Tech-Texas  Inactive   
1017     EMP2017    Charlie  Williams  40.0    Finance-New York    Active   
1018     EMP2018      Alice    Garcia  30.0          HR-Florida  Inactive   
1019     EMP2019      Heidi     Jones  30.0     DevOps-Illinois   Pending   

       Join_Date     Salary                         Email       Phone  \
0 

In [5]:
import sqlite3

conn = sqlite3.connect('employees.db')
df.to_sql('employees', conn, if_exists = 'replace', index = False)
cursor = conn.cursor()
cursor.execute(""" 
SELECT Salary as Salary,
CASE
WHEN Salary < 10000.00 THEN 'Low Income'
WHEN Salary BETWEEN 10000.00 AND 60000.00 THEN 'Medium Income' 
WHEN Salary > 60000.00 THEN 'High Income'
END AS Salary_Category
FROM employees;
""")
try:
    results = cursor.fetchall()
    data = pd.DataFrame(results)
    print(data.head(30))
finally:
    conn.close()

#most employees are high income earners. That should be taken into consideration when the prediction model gives out results.

            0              1
0    59767.65  Medium Income
1    69450.99    High Income
2   109324.61    High Income
3    88642.84    High Income
4    94497.91    High Income
5   115565.82    High Income
6    72081.71    High Income
7    89295.77    High Income
8    97633.68    High Income
9   117975.49    High Income
10   94867.33    High Income
11  100377.65    High Income
12   91926.12    High Income
13  111460.67    High Income
14   96182.56    High Income
15  115023.94    High Income
16   80667.01    High Income
17   91181.61    High Income
18   66025.92    High Income
19   83979.01    High Income
20   56785.16  Medium Income
21   95741.88    High Income
22   63411.31    High Income
23  111214.02    High Income
24  114845.69    High Income
25   78072.82    High Income
26   79610.52    High Income
27   90542.62    High Income
28  114357.10    High Income
29   52781.92  Medium Income


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

def pipeline():

    X = df.drop(['Salary', 'Email', 'Phone', 'First_Name', 'Last_Name', 'Employee_ID', 'Join_Date'], axis = 1)
    y = df['Salary']

    categorical_features = ['Department_Region', 'Join_Year', 'Join_Month', 'Status', 'Remote_Work', 'Performance_Score']
    numerical_features = ['Age']

    preprocessor = ColumnTransformer(transformers = [
            ('cat', StandardScaler(), numerical_features),
            ('num', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
        ])

    X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = 0.3, random_state = 45)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
        ])
    pipeline.fit(X_train, y_train)
    pipeline.predict(X_test)
    return pipeline

print(pipeline())


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', StandardScaler(),
                                                  ['Age']),
                                                 ('num',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Department_Region',
                                                   'Join_Year', 'Join_Month',
                                                   'Status', 'Remote_Work',
                                                   'Performance_Score'])])),
                ('model', LinearRegression())])


In [7]:
import joblib
def create_model_cache():
    modelfile_dir = 'salary_model.joblib'
    joblib.dump(pipeline(), modelfile_dir)

    return joblib.load(modelfile_dir)

    

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from datetime import date

app = FastAPI(title = 'EmployeeDataAnalyZer')

class EmployeeDataInput(BaseModel):
    age : int
    department_region : str
    join_year : int
    join_month : int
    status : str
    remote_work : bool
    performance_score : str

model = create_model_cache()

@app.get('/')
def home_page():
    return {'message' : 'Welcome to the Employee Data AnalyZer!'}

@app.post('/predictions')
def input_data(data : EmployeeDataInput):
    row = {
        "Age": data.age,
        "Department_Region": data.department_region,
        "Join_Year": data.join_year,
        "Join_Month": data.join_month,
        "Status": data.status,
        "Remote_Work": data.remote_work,
        "Performance_Score": data.performance_score,
    }
    input_df = pd.DataFrame([row])
    try:
        predictions = model.predict(input_df)
    except Exception as e:
        raise HTTPException(status_code = 404, detail = f'{e} ERROR HAS OCCURED!')
    
    return {'predictions' : float(predictions)}